In [1]:
!pip install cyvcf2

In [2]:
from cyvcf2 import VCF
import pandas as pd

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
!cp -r "/content/drive/My Drive/Spring 2025/cbb_752" "/content/"
!ls "/content/cbb_752"

cp: cannot open '/content/drive/My Drive/Spring 2025/cbb_752/Discussion 1 Notes.gdoc' for reading: Operation not supported
cp: cannot open '/content/drive/My Drive/Spring 2025/cbb_752/Discussion 1 Summaries.gdoc' for reading: Operation not supported
cp: cannot open '/content/drive/My Drive/Spring 2025/cbb_752/Discussion 2 Summaries.gdoc' for reading: Operation not supported
cp: cannot open '/content/drive/My Drive/Spring 2025/cbb_752/Discussion 3 Summaries.gdoc' for reading: Operation not supported
cp: cannot open '/content/drive/My Drive/Spring 2025/cbb_752/Notes for Lecture Summary.gdoc' for reading: Operation not supported
cp: cannot open '/content/drive/My Drive/Spring 2025/cbb_752/Quiz 1 Discussion Section Review.gdoc' for reading: Operation not supported
cp: cannot open '/content/drive/My Drive/Spring 2025/cbb_752/2023 Quiz Practice.gdoc' for reading: Operation not supported
cp: cannot open '/content/drive/My Drive/Spring 2025/cbb_752/Quiz 1 Review.gdoc' for reading: Operation no

In [5]:
# From the file already processed to sort by mutational burden since it has clinical significance
vep = VCF("cbb_752/ensembl_vep_results.vcf_")

In [6]:
i = 1
for var in vep:
  print(var)
  i += 1

  if i == 10:
    break

22	16050822	.	G	A	.	.	CSQ=A|intergenic_variant|MODIFIER||||||||||||||||G|G/A||||||||||||||||||||||

22	16051249	.	T	C	.	.	CSQ=C|intergenic_variant|MODIFIER||||||||||||||||T|T/C||||||||||||||||||||||

22	16051347	.	G	C	.	.	CSQ=C|intergenic_variant|MODIFIER||||||||||||||||G|G/C||||||||||||||||||||||

22	16051453	.	A	C	.	.	CSQ=C|intergenic_variant|MODIFIER||||||||||||||||A|A/C||||||||||||||||||||||

22	16051497	.	A	G	.	.	CSQ=G|intergenic_variant|MODIFIER||||||||||||||||A|A/G||||||||||||||||||||||

22	16052463	.	T	C	.	.	CSQ=C|intergenic_variant|MODIFIER||||||||||||||||T|T/C||||||||||||||||||||||

22	16052962	.	C	T	.	.	CSQ=T|intergenic_variant|MODIFIER||||||||||||||||C|C/T||||||||||||||||||||||

22	16053659	.	A	C	.	.	CSQ=C|intergenic_variant|MODIFIER|||||||||||||||rs1365946665|A|A/C||||||||||||||||||||||

22	16053791	.	C	A	.	.	CSQ=A|intergenic_variant|MODIFIER||||||||||||||||C|C/A||||||||||||||||||||||



In [7]:
records = []

for var in vep:
    info = var.INFO.get('CSQ')
    # Some variants might not have a CSQ annotation
    if isinstance(info, list):
        info_str = info[0]
    else:
        info_str = info if info is not None else ""

    row = {
        "CHROM": var.CHROM,
        "POS": var.POS,
        "ID": var.ID if var.ID else ".",
        "REF": var.REF,
        "ALT": var.ALT[0] if var.ALT else ".",
        "QUAL": var.QUAL,
        "FILTER": var.FILTER if var.FILTER else ".",
        "INFO": info_str
    }
    records.append(row)

vep_df = pd.DataFrame(records)

In [8]:
# Preview
print(vep_df.head())
print(vep_df.shape)

  CHROM       POS ID REF ALT  QUAL FILTER  \
0    22  16053862  .   C   T  None      .   
1    22  16054454  .   C   T  None      .   
2    22  16055122  .   G   T  None      .   
3    22  16055942  .   C   T  None      .   
4    22  16056126  .   G   A  None      .   

                                                INFO  
0  T|intergenic_variant|MODIFIER|||||||||||||||rs...  
1  T|intergenic_variant|MODIFIER||||||||||||||||C...  
2  T|intergenic_variant|MODIFIER||||||||||||||||G...  
3  T|intergenic_variant|MODIFIER||||||||||||||||C...  
4  A|intergenic_variant|MODIFIER||||||||||||||||G...  
(45583, 8)


In [9]:
vep_df["GENE_SYMBOL"] = vep_df["INFO"].astype(str).str.split("|").str[3]
vep_df["GENE_NAME"] = vep_df["INFO"].astype(str).str.split("|").str[4]
vep_df["IMPACT"] = vep_df["INFO"].astype(str).str.split("|").str[2]

# Define the desired column order
desired_order = ['CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'GENE_SYMBOL', 'GENE_NAME', 'IMPACT', 'INFO']

# Reorder the DataFrame columns
vep_df = vep_df[desired_order]

# Preview
print(vep_df.head())
print(vep_df.shape)

  CHROM       POS ID REF ALT  QUAL FILTER GENE_SYMBOL GENE_NAME    IMPACT  \
0    22  16053862  .   C   T  None      .                        MODIFIER   
1    22  16054454  .   C   T  None      .                        MODIFIER   
2    22  16055122  .   G   T  None      .                        MODIFIER   
3    22  16055942  .   C   T  None      .                        MODIFIER   
4    22  16056126  .   G   A  None      .                        MODIFIER   

                                                INFO  
0  T|intergenic_variant|MODIFIER|||||||||||||||rs...  
1  T|intergenic_variant|MODIFIER||||||||||||||||C...  
2  T|intergenic_variant|MODIFIER||||||||||||||||G...  
3  T|intergenic_variant|MODIFIER||||||||||||||||C...  
4  A|intergenic_variant|MODIFIER||||||||||||||||G...  
(45583, 11)


In [10]:
impact_counts = vep_df.groupby(['GENE_SYMBOL', 'IMPACT']).size().unstack(fill_value=0)

for impact in ['HIGH', 'MODERATE', 'LOW', 'MODIFIER']:
    if impact not in impact_counts.columns:
        impact_counts[impact] = 0

impact_counts['IMPACT_SCORE'] = (
    impact_counts['HIGH'] * 3 +
    impact_counts['MODERATE'] * 2 +
    impact_counts['LOW'] * 1 +
    impact_counts['MODIFIER'] * 0
)

top_genes_by_impact = impact_counts.sort_values(by='IMPACT_SCORE', ascending=False).head(10).reset_index()

print("Top 10 Gene Symbols by Variant Impact Score:")
print(top_genes_by_impact)

Top 10 Gene Symbols by Variant Impact Score:
IMPACT GENE_SYMBOL  HIGH  LOW  MODERATE  MODIFIER  IMPACT_SCORE
0             BRD1     1   10         8       119            29
1           PLXNB2     4    5         5        74            27
2          DENND6B     2    9         5        57            25
3            NUP50     1    5         7       110            22
4           CACNG2     1    4         7       496            21
5            CECR2     1   10         4       364            21
6           EFCAB6     0    6         7       449            20
7           CABIN1     0    2         8       206            18
8              MN1     0    4         7       117            18
9         SERPIND1     2    2         5        40            18
